In [1]:
# Install Feast and required dependencies
!pip install feast
!pip install 'feast[gcp]'  # GCP-specific dependencies
!pip install scikit-learn pandas joblib google-cloud-storage

INFO: pip is looking at multiple versions of uvicorn-worker to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 8.9 MB/s  0:00:006m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 11.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 15.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 14.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 18.6 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.37.0
    Uninstalling uvicorn-0.37.0:
      Successfully uninstalled uvicorn-0.37.0
  Attempting uninstall: tenacity━━━━━━━━━━━━━━━━  0/19 [uvicorn]
    Found existing installation: tenacity 9.1.2m  0/19 [uvicorn]
    Uninstalling tenacity-9.1.2:━━━━━━━━━━━━  0/19 [uvicorn]
      Successfully uninstalled tenacity-9.1.2  0/19 [uvicorn]
  Attempting uninstall: pyd

In [2]:
# Check Feast version
!feast version

Feast SDK Version: "0.56.0"


In [3]:
# Clone the GitHub repository with week_3 resources
!git clone -b week_3 https://github.com/IITMBSMLOps/ga_resources.git

Cloning into 'ga_resources'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 37 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 27.40 KiB | 2.49 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [4]:
# List the contents of the ga_resources folder
!ls -la ga_resources/

total 88
drwxr-xr-x 3 jupyter jupyter  4096 Oct 30 18:54 .
drwxr-xr-x 8 jupyter jupyter  4096 Oct 30 18:54 ..
drwxr-xr-x 8 jupyter jupyter  4096 Oct 30 18:54 .git
-rw-r--r-- 1 jupyter jupyter  4688 Oct 30 18:54 .gitignore
-rw-r--r-- 1 jupyter jupyter 56570 Oct 30 18:54 Modified_Driver_Ranking_Tutorial.ipynb
-rw-r--r-- 1 jupyter jupyter  4298 Oct 30 18:54 README.md
-rw-r--r-- 1 jupyter jupyter  4009 Oct 30 18:54 iris_data_adapted_for_feast.csv


In [5]:
# Read the README to understand the assignment expectations
with open('ga_resources/README.md', 'r') as f:
    readme_content = f.read()
    print(readme_content)

# MLOps Graded Assignment - Week 3 Resources 

# Time-Aware Iris Dataset for Feast Tutorial

## Overview

This directory contains a modified, time-series version of the classic Iris dataset. It has been specifically generated to be compatible with the [Feast feature store](https://feast.dev/) and is intended for use in a hands-on tutorial.

Unlike the original static dataset, this version simulates the tracking of features for a few individual iris plants over a period of time, making it suitable for demonstrating real-world feature store concepts.

---

## The Problem with the Standard Iris Dataset

The standard Iris dataset is a simple table of 150 measurements. While excellent for basic classification tasks, it is unsuitable for demonstrating a feature store because it lacks:

1.  **An Entity**: There is no unique identifier for the object being measured (e.g., a specific plant ID). Feast requires an entity to associate features with.
2.  **Timestamps**: All data exists at a single,

In [6]:
import pandas as pd

# Load the Feast-adapted IRIS dataset
iris_feast_data = pd.read_csv('ga_resources/iris_data_adapted_for_feast.csv')

# Convert timestamp columns to datetime
iris_feast_data['event_timestamp'] = pd.to_datetime(iris_feast_data['event_timestamp'])
iris_feast_data['created_timestamp'] = pd.to_datetime(iris_feast_data['created_timestamp'])

# Display basic info
print("📊 Dataset Shape:", iris_feast_data.shape)
print("\n📋 Column Types:")
print(iris_feast_data.dtypes)
print("\n👀 First 10 Rows:")
print(iris_feast_data.head(10))
print("\n🔍 Unique Iris IDs:", iris_feast_data['iris_id'].unique())
print("\n📅 Date Range:", iris_feast_data['event_timestamp'].min(), "to", iris_feast_data['event_timestamp'].max())

📊 Dataset Shape: (45, 8)

📋 Column Types:
event_timestamp      datetime64[ns]
iris_id                       int64
sepal_length                float64
sepal_width                 float64
petal_length                float64
petal_width                 float64
species                      object
created_timestamp    datetime64[ns]
dtype: object

👀 First 10 Rows:
             event_timestamp  iris_id  sepal_length  sepal_width  \
0 2025-09-17 10:40:17.102131     1001          5.52         2.53   
1 2025-09-18 10:40:17.102131     1001          5.50         2.24   
2 2025-09-19 10:40:17.102131     1001          5.55         2.47   
3 2025-09-20 10:40:17.102131     1001          5.45         2.37   
4 2025-09-21 10:40:17.102131     1001          5.65         2.52   
5 2025-09-22 10:40:17.102131     1001          5.59         2.19   
6 2025-09-23 10:40:17.102131     1001          5.38         2.35   
7 2025-09-24 10:40:17.102131     1001          5.62         2.34   
8 2025-09-25 10:40:17.1021

In [7]:
# Create a directory for our Feast project
!mkdir -p feast_iris_project
%cd feast_iris_project

# Initialize Feast project
!feast init -t local iris_feature_store

/home/jupyter/ga_resources/feast_iris_project


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]



Creating a new Feast repository in /home/jupyter/ga_resources/feast_iris_project/iris_feature_store.



In [8]:
# Navigate into the Feast project directory
%cd iris_feature_store

# List all files
!ls -la

/home/jupyter/ga_resources/feast_iris_project/iris_feature_store
total 20
drwxr-xr-x 3 jupyter jupyter 4096 Oct 30 18:59 .
drwxr-xr-x 3 jupyter jupyter 4096 Oct 30 18:59 ..
-rw-r--r-- 1 jupyter jupyter  463 Oct 30 18:46 .gitignore
-rw-r--r-- 1 jupyter jupyter 2516 Oct 30 18:46 README.md
-rw-r--r-- 1 jupyter jupyter    0 Oct 30 18:46 __init__.py
drwxr-xr-x 4 jupyter jupyter 4096 Oct 30 18:59 feature_repo


In [9]:
# Copy the IRIS dataset to the current Feast project directory
!cp /home/jupyter/ga_resources/iris_data_adapted_for_feast.csv .

# Verify it's copied
!ls -lh iris_data_adapted_for_feast.csv

cp: cannot stat '/home/jupyter/ga_resources/iris_data_adapted_for_feast.csv': No such file or directory
ls: cannot access 'iris_data_adapted_for_feast.csv': No such file or directory


In [10]:
# First, let's see where we are
!pwd

# Copy from the correct path
!cp ../../iris_data_adapted_for_feast.csv .

# Verify it's copied
!ls -lh iris_data_adapted_for_feast.csv

/home/jupyter/ga_resources/feast_iris_project/iris_feature_store
cp: cannot stat '../../iris_data_adapted_for_feast.csv': No such file or directory
ls: cannot access 'iris_data_adapted_for_feast.csv': No such file or directory


In [11]:
# Let's find where the file actually is
!find /home/jupyter -name "iris_data_adapted_for_feast.csv" -type f 2>/dev/null

/home/jupyter/ga_resources/ga_resources/iris_data_adapted_for_feast.csv


In [12]:
# Copy from the correct absolute path
!cp /home/jupyter/ga_resources/ga_resources/iris_data_adapted_for_feast.csv .

# Verify it's copied successfully
!ls -lh iris_data_adapted_for_feast.csv

# Also show first few lines to confirm
!head -5 iris_data_adapted_for_feast.csv

-rw-r--r-- 1 jupyter jupyter 4.0K Oct 30 19:33 iris_data_adapted_for_feast.csv
event_timestamp,iris_id,sepal_length,sepal_width,petal_length,petal_width,species,created_timestamp
2025-09-17 10:40:17.102131,1001,5.52,2.53,3.86,1.13,versicolor,2025-10-02 10:40:17.172178
2025-09-18 10:40:17.102131,1001,5.5,2.24,3.6,1.08,versicolor,2025-10-02 10:40:17.172178
2025-09-19 10:40:17.102131,1001,5.55,2.47,3.75,1.08,versicolor,2025-10-02 10:40:17.172178
2025-09-20 10:40:17.102131,1001,5.45,2.37,3.92,1.2,versicolor,2025-10-02 10:40:17.172178


In [13]:
# Check your current GCP project
!gcloud config get-value project

# List your buckets (to see the bucket name)
!gsutil ls

corded-forge-475015-k6


To take a quick anonymous survey, run:
  $ gcloud survey

gs://corded-forge-475015-k6-bucket/
gs://mlops-week-1-bucket/


In [14]:
# Read the current feature_store.yaml
with open('feature_store.yaml', 'r') as f:
    config_content = f.read()
    print(config_content)

FileNotFoundError: [Errno 2] No such file or directory: 'feature_store.yaml'

In [15]:
# Check what's in the feature_repo directory
!ls -la feature_repo/

# Also check the current directory more carefully
!ls -la

total 36
drwxr-xr-x 4 jupyter jupyter 4096 Oct 30 18:59 .
drwxr-xr-x 3 jupyter jupyter 4096 Oct 30 19:33 ..
-rw-r--r-- 1 jupyter jupyter    0 Oct 30 18:46 __init__.py
drwxr-xr-x 2 jupyter jupyter 4096 Oct 30 18:46 __pycache__
drwxr-xr-x 2 jupyter jupyter 4096 Oct 30 18:59 data
-rw-r--r-- 1 jupyter jupyter 5255 Oct 30 18:59 example_repo.py
-rw-r--r-- 1 jupyter jupyter  543 Oct 30 18:59 feature_store.yaml
-rw-r--r-- 1 jupyter jupyter 4378 Oct 30 18:46 test_workflow.py
total 24
drwxr-xr-x 3 jupyter jupyter 4096 Oct 30 19:33 .
drwxr-xr-x 3 jupyter jupyter 4096 Oct 30 18:59 ..
-rw-r--r-- 1 jupyter jupyter  463 Oct 30 18:46 .gitignore
-rw-r--r-- 1 jupyter jupyter 2516 Oct 30 18:46 README.md
-rw-r--r-- 1 jupyter jupyter    0 Oct 30 18:46 __init__.py
drwxr-xr-x 4 jupyter jupyter 4096 Oct 30 18:59 feature_repo
-rw-r--r-- 1 jupyter jupyter 4009 Oct 30 19:33 iris_data_adapted_for_feast.csv


In [16]:
# Read the feature_store.yaml from current directory
!cat feature_store.yaml

cat: feature_store.yaml: No such file or directory


In [17]:
# Create feature_store.yaml with GCS configuration
feature_store_config = """project: iris_feature_store
provider: local
registry:
  registry_type: gcs
  path: gs://corded-forge-475015-k6-bucket/feast/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 2
"""

# Write the configuration file
with open('feature_store.yaml', 'w') as f:
    f.write(feature_store_config)

print("✅ feature_store.yaml created successfully!")
print("\n📄 Configuration:")
print(feature_store_config)

✅ feature_store.yaml created successfully!

📄 Configuration:
project: iris_feature_store
provider: local
registry:
  registry_type: gcs
  path: gs://corded-forge-475015-k6-bucket/feast/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 2



In [18]:
# Create the feature definitions file
feature_definitions = '''from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String

# Define the entity (iris plant)
iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    description="Unique identifier for each iris plant"
)

# Define the data source (our CSV file)
iris_source = FileSource(
    path="iris_data_adapted_for_feast.csv",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

# Define the feature view with all iris features
iris_features = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=30),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String),
    ],
    source=iris_source,
    online=True,
)
'''

# Write to feature_repo.py
with open('feature_repo.py', 'w') as f:
    f.write(feature_definitions)

print("✅ feature_repo.py created successfully!")
print("\n📄 Feature Definitions:")
print(feature_definitions)

✅ feature_repo.py created successfully!

📄 Feature Definitions:
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String

# Define the entity (iris plant)
iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    description="Unique identifier for each iris plant"
)

# Define the data source (our CSV file)
iris_source = FileSource(
    path="iris_data_adapted_for_feast.csv",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

# Define the feature view with all iris features
iris_features = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=30),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String),
    ],
    source=iris_source,


In [19]:
# Apply the feature definitions
!feast apply

/opt/conda/lib/python3.10/site-packages/feast/repo_config.py:278: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(
/home/jupyter/ga_resources/feast_iris_project/iris_feature_store/feature_repo/example_repo.py:27: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'driver'.
  driver = Entity(name="driver", join_keys=["driver_id"])
Traceback (most recent call last):
  File "/opt/conda/bin/feast", line 7, in <module>
    sys.exit(cli())
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1161, in __call__
    return self.main(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1082, in main
    rv = self.invoke(ctx)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1697, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
  File "/opt/

In [20]:
# Create the corrected feature_store.yaml
feature_store_config = """project: iris_feature_store
provider: local
registry: gs://corded-forge-475015-k6-bucket/feast/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3
"""

# Write the corrected configuration file
with open('feature_store.yaml', 'w') as f:
    f.write(feature_store_config)

print("✅ feature_store.yaml updated successfully!")
print("\n📄 New Configuration:")
print(feature_store_config)

✅ feature_store.yaml updated successfully!

📄 New Configuration:
project: iris_feature_store
provider: local
registry: gs://corded-forge-475015-k6-bucket/feast/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3



In [21]:
# Apply the feature definitions
!feast apply

/home/jupyter/ga_resources/feast_iris_project/iris_feature_store/feature_repo/example_repo.py:27: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'driver'.
  driver = Entity(name="driver", join_keys=["driver_id"])
Applying changes for project iris_feature_store
/opt/conda/lib/python3.10/site-packages/feast/feature_store.py:583: RuntimeWarning: On demand feature view is an experimental feature. This API is stable, but the functionality does not scale well for offline retrieval
  warnings.warn(
Traceback (most recent call last):
  File "/opt/conda/bin/feast", line 7, in <module>
    sys.exit(cli())
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1161, in __call__
    return self.main(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1082, in main
    rv = self.invoke(ctx)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1697, in invoke
    retur

In [22]:
# Remove or rename the example files so they don't interfere
!mv feature_repo/example_repo.py feature_repo/example_repo.py.bak
!mv test_workflow.py test_workflow.py.bak 2>/dev/null || true

# Verify they're renamed
!ls -la feature_repo/
!ls -la *.py

total 36
drwxr-xr-x 4 jupyter jupyter 4096 Oct 30 20:31 .
drwxr-xr-x 3 jupyter jupyter 4096 Oct 30 20:00 ..
-rw-r--r-- 1 jupyter jupyter    0 Oct 30 18:46 __init__.py
drwxr-xr-x 2 jupyter jupyter 4096 Oct 30 18:46 __pycache__
drwxr-xr-x 2 jupyter jupyter 4096 Oct 30 18:59 data
-rw-r--r-- 1 jupyter jupyter 5255 Oct 30 18:59 example_repo.py.bak
-rw-r--r-- 1 jupyter jupyter  543 Oct 30 18:59 feature_store.yaml
-rw-r--r-- 1 jupyter jupyter 4378 Oct 30 18:46 test_workflow.py
-rw-r--r-- 1 jupyter jupyter   0 Oct 30 18:46 __init__.py
-rw-r--r-- 1 jupyter jupyter 955 Oct 30 20:00 feature_repo.py


In [23]:
# Apply the feature definitions
!feast apply

No project found in the repository. Using project name iris_feature_store defined in feature_store.yaml
Applying changes for project iris_feature_store
Created project iris_feature_store

No changes to infrastructure


In [24]:
# Move our feature definitions to the feature_repo directory
!mv feature_repo.py feature_repo/

# Verify it's in the right place
!ls -la feature_repo/

# Show the content to confirm
!cat feature_repo/feature_repo.py

total 40
drwxr-xr-x 4 jupyter jupyter 4096 Oct 30 20:35 .
drwxr-xr-x 3 jupyter jupyter 4096 Oct 30 20:35 ..
-rw-r--r-- 1 jupyter jupyter    0 Oct 30 18:46 __init__.py
drwxr-xr-x 2 jupyter jupyter 4096 Oct 30 18:46 __pycache__
drwxr-xr-x 2 jupyter jupyter 4096 Oct 30 18:59 data
-rw-r--r-- 1 jupyter jupyter 5255 Oct 30 18:59 example_repo.py.bak
-rw-r--r-- 1 jupyter jupyter  955 Oct 30 20:00 feature_repo.py
-rw-r--r-- 1 jupyter jupyter  543 Oct 30 18:59 feature_store.yaml
-rw-r--r-- 1 jupyter jupyter 4378 Oct 30 18:46 test_workflow.py
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String

# Define the entity (iris plant)
iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    description="Unique identifier for each iris plant"
)

# Define the data source (our CSV file)
iris_source = FileSource(
    path="iris_data_adapted_for_feast.csv",
    timestamp_field="event_timestamp",
    created_timestamp_c

In [25]:
# Update feature_repo.py with correct CSV path
feature_definitions = '''from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String

# Define the entity (iris plant)
iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    description="Unique identifier for each iris plant"
)

# Define the data source (our CSV file) - using correct relative path
iris_source = FileSource(
    path="../iris_data_adapted_for_feast.csv",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

# Define the feature view with all iris features
iris_features = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=30),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String),
    ],
    source=iris_source,
    online=True,
)
'''

# Write to feature_repo/feature_repo.py
with open('feature_repo/feature_repo.py', 'w') as f:
    f.write(feature_definitions)

print("✅ feature_repo.py updated with correct path!")

✅ feature_repo.py updated with correct path!


In [26]:
# Apply the feature definitions
!feast apply

/home/jupyter/ga_resources/feast_iris_project/iris_feature_store/feature_repo/feature_repo.py:6: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_id'.
  iris = Entity(
No project found in the repository. Using project name iris_feature_store defined in feature_store.yaml
Applying changes for project iris_feature_store
Traceback (most recent call last):
  File "/opt/conda/bin/feast", line 7, in <module>
    sys.exit(cli())
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1161, in __call__
    return self.main(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1082, in main
    rv = self.invoke(ctx)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1697, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1443, in invoke
    return ctx.invoke(self.callback, 

In [27]:
# Update feature_repo.py with absolute path
feature_definitions = '''from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String, Int32

# Define the entity (iris plant) with value_type
iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    value_type=Int32,
    description="Unique identifier for each iris plant"
)

# Define the data source (our CSV file) - using absolute path
iris_source = FileSource(
    path="/home/jupyter/ga_resources/feast_iris_project/iris_feature_store/iris_data_adapted_for_feast.csv",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

# Define the feature view with all iris features
iris_features = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=30),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String),
    ],
    source=iris_source,
    online=True,
)
'''

# Write to feature_repo/feature_repo.py
with open('feature_repo/feature_repo.py', 'w') as f:
    f.write(feature_definitions)

print("✅ feature_repo.py updated with absolute path!")
print("   Also added value_type to fix the deprecation warning!")

✅ feature_repo.py updated with absolute path!
   Also added value_type to fix the deprecation warning!


In [28]:
# Apply the feature definitions
!feast apply

Traceback (most recent call last):
  File "/opt/conda/bin/feast", line 7, in <module>
    sys.exit(cli())
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1161, in __call__
    return self.main(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1082, in main
    rv = self.invoke(ctx)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1697, in invoke
    return _process_result(sub_ctx.command.invoke(sub_ctx))
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 1443, in invoke
    return ctx.invoke(self.callback, **ctx.params)
  File "/opt/conda/lib/python3.10/site-packages/click/core.py", line 788, in invoke
    return __callback(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/click/decorators.py", line 33, in new_func
    return f(get_current_context(), *args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/feast/cli/cli.py", line 272, in apply_total_command
    apply_to

In [29]:
# Update feature_repo.py with correct value_type
feature_definitions = '''from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource, ValueType
from feast.types import Float32, String

# Define the entity (iris plant) with correct ValueType
iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    value_type=ValueType.INT64,
    description="Unique identifier for each iris plant"
)

# Define the data source (our CSV file) - using absolute path
iris_source = FileSource(
    path="/home/jupyter/ga_resources/feast_iris_project/iris_feature_store/iris_data_adapted_for_feast.csv",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

# Define the feature view with all iris features
iris_features = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=30),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String),
    ],
    source=iris_source,
    online=True,
)
'''

# Write to feature_repo/feature_repo.py
with open('feature_repo/feature_repo.py', 'w') as f:
    f.write(feature_definitions)

print("✅ feature_repo.py updated with correct ValueType!")

✅ feature_repo.py updated with correct ValueType!


In [30]:
# Apply the feature definitions
!feast apply

No project found in the repository. Using project name iris_feature_store defined in feature_store.yaml
Applying changes for project iris_feature_store
Created entity iris_id
Created feature view iris_features

Created sqlite table iris_feature_store_iris_features



In [31]:
# Materialize features from the offline store (CSV) to the online store (SQLite)
# Using the date range from our dataset: Sept 17 - Oct 1, 2025
!feast materialize 2025-09-17T00:00:00 2025-10-02T00:00:00

Materializing 1 feature views from 2025-09-17 00:00:00+00:00 to 2025-10-02 00:00:00+00:00 into the sqlite online store.

iris_features:
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/dask/backends.py", line 140, in wrapper
    return func(*args, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/dask/dataframe/dask_expr/_collection.py", line 5381, in read_parquet
    ReadParquetFSSpec(
  File "/opt/conda/lib/python3.10/site-packages/dask/_expr.py", line 878, in __new__
    inst = super().__new__(cls, *args, _determ_token=_determ_token, **kwargs)
  File "/opt/conda/lib/python3.10/site-packages/dask/_expr.py", line 72, in __new__
    inst._name
  File "/opt/conda/lib/python3.10/functools.py", line 981, in __get__
    val = self.func(instance)
  File "/opt/conda/lib/python3.10/site-packages/dask/dataframe/dask_expr/io/parquet.py", line 790, in _name
    return self._funcname + "-" + self.deterministic_token
  File "/opt/conda/lib/python3.10/site-

In [32]:
import pandas as pd

# Read the CSV file
iris_data = pd.read_csv('iris_data_adapted_for_feast.csv')

# Convert timestamp columns to datetime
iris_data['event_timestamp'] = pd.to_datetime(iris_data['event_timestamp'])
iris_data['created_timestamp'] = pd.to_datetime(iris_data['created_timestamp'])

# Save as Parquet file
iris_data.to_parquet('iris_data_adapted_for_feast.parquet', index=False)

print("✅ Converted CSV to Parquet format!")
print(f"📊 Parquet file created with {len(iris_data)} rows")

# Verify the file exists
!ls -lh iris_data_adapted_for_feast.parquet

✅ Converted CSV to Parquet format!
📊 Parquet file created with 45 rows
-rw-r--r-- 1 jupyter jupyter 6.4K Oct 30 20:40 iris_data_adapted_for_feast.parquet


In [33]:
# Update feature_repo.py to use Parquet file
feature_definitions = '''from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource, ValueType
from feast.types import Float32, String

# Define the entity (iris plant) with correct ValueType
iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    value_type=ValueType.INT64,
    description="Unique identifier for each iris plant"
)

# Define the data source (our Parquet file) - using absolute path
iris_source = FileSource(
    path="/home/jupyter/ga_resources/feast_iris_project/iris_feature_store/iris_data_adapted_for_feast.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

# Define the feature view with all iris features
iris_features = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=30),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String),
    ],
    source=iris_source,
    online=True,
)
'''

# Write to feature_repo/feature_repo.py
with open('feature_repo/feature_repo.py', 'w') as f:
    f.write(feature_definitions)

print("✅ feature_repo.py updated to use Parquet file!")

✅ feature_repo.py updated to use Parquet file!


In [34]:
# Apply the updated feature definitions
!feast apply

No project found in the repository. Using project name iris_feature_store defined in feature_store.yaml
Applying changes for project iris_feature_store
Updated feature view iris_features
	batch_source: type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created_timestamp"
file_options {
  uri: "/home/jupyter/ga_resources/feast_iris_project/iris_feature_store/iris_data_adapted_for_feast.csv"
}
data_source_class_type: "feast.infra.offline_stores.file_source.FileSource"
name: "/home/jupyter/ga_resources/feast_iris_project/iris_feature_store/iris_data_adapted_for_feast.csv"
meta {
  created_timestamp {
    seconds: 1761856761
    nanos: 133826000
  }
  last_updated_timestamp {
    seconds: 1761856763
    nanos: 70158000
  }
}
 -> type: BATCH_FILE
timestamp_field: "event_timestamp"
created_timestamp_column: "created_timestamp"
file_options {
  uri: "/home/jupyter/ga_resources/feast_iris_project/iris_feature_store/iris_data_adapted_for_feast.parquet"
}
data_source_

In [36]:
# Materialize features from the offline store (Parquet) to the online store (SQLite)
!feast materialize 2025-09-17T00:00:00 2025-10-02T00:00:00

Materializing 1 feature views from 2025-09-17 00:00:00+00:00 to 2025-10-02 00:00:00+00:00 into the sqlite online store.

iris_features:


In [37]:
from feast import FeatureStore
import pandas as pd

# Initialize the feature store
store = FeatureStore(repo_path=".")

# Create entity dataframe with iris_ids we want to get features for
entity_df = pd.DataFrame({
    "iris_id": [1001, 1002, 1003]  # Our 3 iris plants
})

# Fetch features from online store
online_features = store.get_online_features(
    entity_rows=entity_df.to_dict('records'),
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
        "iris_features:species",
    ],
).to_df()

print("🎉 Successfully fetched features from online store!")
print("\n📊 Features for our 3 iris plants:")
print(online_features)

🎉 Successfully fetched features from online store!

📊 Features for our 3 iris plants:
   iris_id  sepal_length  sepal_width  petal_length  petal_width     species
0     1001          5.45         2.36          3.84         1.09  versicolor
1     1002          4.84         2.90          1.29         0.20      setosa
2     1003          4.85         3.40          1.19         0.29      setosa


In [38]:
from feast import FeatureStore
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Initialize feature store
store = FeatureStore(repo_path=".")

# Create entity dataframe with all timestamps and iris_ids
# We'll use the original data to create entity rows for historical features
iris_data = pd.read_parquet('iris_data_adapted_for_feast.parquet')

# Create entity dataframe with event_timestamp and iris_id
entity_df = iris_data[['event_timestamp', 'iris_id', 'species']].copy()

print("📊 Fetching historical features from Feast...")

# Get historical features for training
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
).to_df()

print(f"✅ Retrieved {len(training_df)} training samples")
print("\n👀 Sample training data:")
print(training_df.head())

# Prepare features and labels
X = training_df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
y = entity_df['species']  # Use original labels

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"\n🔧 Training Decision Tree model...")
print(f"   Training samples: {len(X_train)}")
print(f"   Test samples: {len(X_test)}")

# Train model
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Model trained successfully!")
print(f"🎯 Accuracy: {accuracy:.3f}")
print(f"\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

# Save the model
joblib.dump(model, 'iris_feast_model.joblib')
print("\n💾 Model saved as 'iris_feast_model.joblib'")

📊 Fetching historical features from Feast...
✅ Retrieved 45 training samples

👀 Sample training data:
                   event_timestamp  iris_id     species  sepal_length  \
0 2025-09-17 10:40:17.102131+00:00     1001  versicolor          5.52   
1 2025-09-17 10:40:17.102131+00:00     1003      setosa          5.09   
2 2025-09-17 10:40:17.102131+00:00     1002      setosa          4.92   
3 2025-09-18 10:40:17.102131+00:00     1001  versicolor          5.50   
4 2025-09-18 10:40:17.102131+00:00     1002      setosa          5.05   

   sepal_width  petal_length  petal_width  
0         2.53          3.86         1.13  
1         3.42          1.34         0.22  
2         3.05          1.43         0.29  
3         2.24          3.60         1.08  
4         2.94          1.53         0.31  

🔧 Training Decision Tree model...
   Training samples: 31
   Test samples: 14

✅ Model trained successfully!
🎯 Accuracy: 0.714

📊 Classification Report:
              precision    recall  f1-sco

In [39]:
from feast import FeatureStore
import pandas as pd
import joblib

# Load the trained model
model = joblib.load('iris_feast_model.joblib')

# Initialize feature store
store = FeatureStore(repo_path=".")

print("🔮 Making predictions using online features from Feast!\n")

# Test with our 3 iris plants
test_iris_ids = [1001, 1002, 1003]

for iris_id in test_iris_ids:
    # Fetch features from online store
    entity_df = pd.DataFrame({"iris_id": [iris_id]})
    
    online_features = store.get_online_features(
        entity_rows=entity_df.to_dict('records'),
        features=[
            "iris_features:sepal_length",
            "iris_features:sepal_width",
            "iris_features:petal_length",
            "iris_features:petal_width",
        ],
    ).to_df()
    
    # Prepare features for prediction
    X = online_features[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
    
    # Make prediction
    prediction = model.predict(X)[0]
    
    print(f"🌸 Iris Plant {iris_id}:")
    print(f"   Features: sepal({X['sepal_length'].values[0]:.2f}×{X['sepal_width'].values[0]:.2f}), "
          f"petal({X['petal_length'].values[0]:.2f}×{X['petal_width'].values[0]:.2f})")
    print(f"   ✅ Predicted Species: {prediction}\n")

print("🎉 Real-time inference complete using Feast online store!")

🔮 Making predictions using online features from Feast!

🌸 Iris Plant 1001:
   Features: sepal(5.45×2.36), petal(3.84×1.09)
   ✅ Predicted Species: setosa

🌸 Iris Plant 1002:
   Features: sepal(4.84×2.90), petal(1.29×0.20)
   ✅ Predicted Species: setosa

🌸 Iris Plant 1003:
   Features: sepal(4.85×3.40), petal(1.19×0.29)
   ✅ Predicted Species: setosa

🎉 Real-time inference complete using Feast online store!


In [40]:
# Check if the registry exists in GCS
!gsutil ls gs://corded-forge-475015-k6-bucket/feast/

# Show registry details
!gsutil ls -lh gs://corded-forge-475015-k6-bucket/feast/registry.db

gs://corded-forge-475015-k6-bucket/feast/registry.db
  1.34 KiB  2025-10-30T20:43:20Z  gs://corded-forge-475015-k6-bucket/feast/registry.db
TOTAL: 1 objects, 1370 bytes (1.34 KiB)
